In [1]:
import json
import math
from pathlib import Path

import numpy as np
import yaml
from sympy.physics.quantum.cg import CG

EPS = 1e-8

# Functions

In [2]:
# -----------------------------------------------------------------------------
# Minimal, self-contained copies/adaptations of the TF-PWA math used on the
# Psi(4040) chain. These functions intentionally do not import tf_pwa.
# -----------------------------------------------------------------------------


def dot3(a, b):
    return np.sum(a * b, axis=-1)


def norm3(a):
    return np.linalg.norm(a, axis=-1)


def unit(v):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / n


def cross_unit(a, b):
    # Copied from TF-PWA Vector3.cross_unit in spirit: if the cross product is
    # degenerate, bias the second vector before normalizing.
    cro = np.cross(a, b)
    norm = np.linalg.norm(cro, axis=-1, keepdims=True)
    mask = norm < EPS
    bias_other = np.ones_like(norm) + b
    cro = np.where(mask, np.cross(a, bias_other), cro)
    return unit(cro)


def angle_from(v, x_axis, y_axis):
    # Angle of vector v measured in the coordinate basis (x_axis, y_axis).
    return np.arctan2(dot3(v, y_axis), dot3(v, x_axis))


def angle_zx_z_getx(z1, x1, z2):
    # Adapted from TF-PWA EulerAngle.angle_zx_z_getx.
    # Physical meaning: construct the Euler rotation from an old frame (z1,x1)
    # to a new helicity frame whose z-axis follows a daughter momentum z2.
    u_z1 = unit(z1)
    u_z2 = unit(z2)
    u_y1 = cross_unit(z1, x1)
    u_x1 = cross_unit(u_y1, z1)
    u_yr = cross_unit(z1, z2)
    u_xr = cross_unit(u_yr, z1)
    alpha = angle_from(u_xr, u_x1, u_y1)
    beta = angle_from(u_z2, u_z1, u_xr)
    gamma = np.zeros_like(beta)
    u_x2 = cross_unit(u_yr, u_z2)
    return {"alpha": alpha, "beta": beta, "gamma": gamma}, u_x2


def invariant_mass(p4):
    # TF-PWA LorentzVector.M with metric (+,-,-,-).
    m2 = p4[..., 0] ** 2 - np.sum(p4[..., 1:] ** 2, axis=-1)
    return np.sqrt(np.abs(m2))


def boost_vector(p4):
    return p4[..., 1:] / p4[..., 0:1]


def boost(p4, beta):
    # Adapted from TF-PWA LorentzVector.boost.
    beta2 = np.sum(beta * beta, axis=-1)
    gamma = 1.0 / np.sqrt(1.0 - beta2)
    bp = np.sum(beta * p4[..., 1:], axis=-1)
    gamma2 = np.where(beta2 > EPS, (gamma - 1.0) / beta2, 0.0)
    spatial = p4[..., 1:]
    spatial = spatial + gamma2[..., None] * bp[..., None] * beta
    spatial = spatial + gamma[..., None] * p4[..., 0:1] * beta
    energy = (gamma * (p4[..., 0] + bp))[..., None]
    return np.concatenate([energy, spatial], axis=-1)


def rest_vector(core_p4, other_p4):
    # Boost other_p4 into the rest frame of core_p4.
    return boost(other_p4, -boost_vector(core_p4))


def get_relative_p2(m0, m1, m2):
    # Two-body breakup momentum squared, copied from TF-PWA formula.get_relative_p2.
    return ((m0 * m0 - (m1 + m2) ** 2) * (m0 * m0 - (m1 - m2) ** 2)) / (4 * m0 * m0)


def bprime_polynomial(l, z):
    # Blatt-Weisskopf polynomials used by TF-PWA up to the orders needed here.
    coeff = {
        0: [1.0],
        1: [1.0, 1.0],
        2: [1.0, 3.0, 9.0],
        3: [1.0, 6.0, 45.0, 225.0],
    }
    return np.polyval(coeff[int(l)], z)


def bprime_q2(l, q2, q02, d=3.0):
    # Adapted from TF-PWA breit_wigner.Bprime_q2.
    z0 = q02 * d**2
    z = q2 * d**2
    ratio = bprime_polynomial(l, z0) / bprime_polynomial(l, z)
    return np.sqrt(np.where(ratio > 0, ratio, 1.0))


def barrier_factor2(l, mass, q2, q02, d=3.0, barrier_factor_norm=True):
    # Adapted from HelicityDecay.get_barrier_factor2 for this chain.
    # Physical meaning: centrifugal-barrier factor q^L B'_L(q,q0,d), normalized by q0^L.
    tmp = q2 ** (l / 2) * bprime_q2(l, q2, q02, d)
    if barrier_factor_norm:
        tmp = tmp / np.abs(q02) ** (l / 2)
    return tmp.reshape(-1, 1)


def gamma_running(m, gamma0, q, q0, l, m0, d=3.0):
    # Adapted from TF-PWA breit_wigner.Gamma.
    qq0 = np.where(q0 > 1e-15, (q / q0) ** (2 * l + 1), 1.0)
    mm0 = m0 / m
    bp = (np.sqrt(bprime_polynomial(l, (q0 * d) ** 2)) / np.sqrt(bprime_polynomial(l, (q * d) ** 2))) ** 2
    return gamma0 * qq0 * mm0 * bp


def bwr(m, m0, gamma0, q, q0, l, d=3.0):
    # Adapted from TF-PWA breit_wigner.BWR.
    # Physical meaning: relativistic Breit-Wigner propagator with running width.
    gamma_m = gamma_running(m, gamma0, q, q0, l, m0, d)
    x = m0 * m0 - m * m
    y = m0 * gamma_m
    denom = x * x + y * y
    return x / denom + 1j * y / denom


def small_d_weight(j2):
    # Copied/adapted from TF-PWA dfun.small_d_weight. j2 means 2*j.
    ret = np.zeros((j2 + 1, j2 + 1, j2 + 1))

    def half_factorial(x):
        return math.factorial(x >> 1)

    for m in range(-j2, j2 + 1, 2):
        for n in range(-j2, j2 + 1, 2):
            for k in range(max(0, n - m), min(j2 - m, j2 + n) + 1, 2):
                ell = (2 * k + (m - n)) // 2
                sign = (-1) ** ((k + m - n) // 2)
                val = sign * math.sqrt(
                    half_factorial(j2 + m)
                    * half_factorial(j2 - m)
                    * half_factorial(j2 + n)
                    * half_factorial(j2 - n)
                )
                val /= (
                    half_factorial(j2 - m - k)
                    * half_factorial(j2 + n - k)
                    * half_factorial(k + m - n)
                    * half_factorial(k)
                )
                ret[ell][(m + j2) // 2][(n + j2) // 2] = val
    return ret


def small_d_matrix(theta, j2):
    theta = np.asarray(theta)
    powers = np.arange(0, j2 + 1).reshape(1, -1)
    half_theta = 0.5 * theta.reshape(-1, 1)
    sc = (np.sin(half_theta) ** powers) * (np.cos(half_theta) ** (j2 - powers))
    weights = small_d_weight(j2).reshape(j2 + 1, (j2 + 1) * (j2 + 1))
    return (sc @ weights).reshape(-1, j2 + 1, j2 + 1)


def d_matrix_conj(alpha, beta, gamma, j2):
    # Adapted from TF-PWA dfun.D_matrix_conj.
    # Physical meaning: conjugated Wigner D matrix D^J*(alpha,beta,gamma).
    m = np.arange(-j2 / 2, j2 / 2 + 1, 1).reshape(1, -1)
    d_small = small_d_matrix(beta, j2)
    exp_alpha = np.exp(1j * alpha.reshape(-1, 1) * m).reshape(-1, j2 + 1, 1)
    exp_gamma = np.exp(1j * gamma.reshape(-1, 1) * m).reshape(-1, 1, j2 + 1)
    return exp_alpha * exp_gamma * d_small.astype(complex)


def dfun_delta_v2(d, ja, la, lb, lc=(0,)):
    # Adapted from TF-PWA dfun.Dfun_delta_v2.
    ln = int(2 * ja + 1 + 0.1)
    idx = []
    max_idx = ln * ln
    for la_i in la:
        for lb_i in lb:
            for lc_i in lc:
                delta = lb_i - lc_i
                if abs(delta) <= ja:
                    idx.append(int((la_i + ja) * ln + delta + ja + 0.1))
                else:
                    idx.append(max_idx)
    flat = d.reshape(-1, ln * ln)
    padded = np.pad(flat, ((0, 0), (0, 1)))
    return padded[:, idx].reshape(-1, len(la), len(lb), len(lc))


def get_d_matrix_lambda(angle, ja, la, lb, lc=None):
    # Adapted from TF-PWA dfun.get_D_matrix_lambda.
    d = d_matrix_conj(angle["alpha"], angle["beta"], angle.get("gamma", np.zeros_like(angle["beta"])), int(2 * ja + 0.1))
    if lc is None:
        return dfun_delta_v2(d, ja, la, lb, (0,)).reshape(-1, len(la), len(lb))
    return dfun_delta_v2(d, ja, la, lb, lc)


def cg_coef(j1, j2, m1, m2, j, m):
    # Same convention as TF-PWA cg.cg_coef when SymPy is available.
    return float(CG(j1, m1, j2, m2, j, m).doit().evalf())


def cg_matrix(ja, jb, jc, ls_list, out_spins):
    # Adapted from HelicityDecay._get_cg_matrix.
    # Physical meaning: LS-to-helicity transformation matrix.
    ret = np.zeros((len(ls_list), len(out_spins[0]), len(out_spins[1])))
    for i, (ell, spin) in enumerate(ls_list):
        for ib, lambda_b in enumerate(out_spins[0]):
            for ic, lambda_c in enumerate(out_spins[1]):
                ret[i, ib, ic] = (
                    math.sqrt(2 * ell + 1)
                    / math.sqrt(2 * ja + 1)
                    * cg_coef(jb, jc, lambda_b, -lambda_c, spin, lambda_b - lambda_c)
                    * cg_coef(ell, spin, 0, lambda_b - lambda_c, ja, lambda_b - lambda_c)
                )
    return ret


def helicity_decay_amp(core_j, out_js, core_spins, out_spins, ls_list, g_ls, angle, mass_core, q2, q02, has_barrier=True, barrier_norm=True):
    # Adapted from HelicityDecay.get_amp -> get_helicity_amp -> get_ls_amp -> get_D_matrix_term.
    # Physical meaning: tensor amplitude for one two-body decay vertex.
    if has_barrier:
        ell = ls_list[0][0]
        bf = barrier_factor2(ell, mass_core, q2, q02, d=3.0, barrier_factor_norm=barrier_norm)
        m_dep = np.array([[g_ls]], dtype=complex) * bf.astype(complex)
    else:
        m_dep = np.array([[g_ls]], dtype=complex)

    cg = cg_matrix(core_j, out_js[0], out_js[1], ls_list, out_spins).astype(complex)
    h = np.sum(m_dep.reshape(-1, len(ls_list), 1, 1) * cg.reshape(len(ls_list), len(out_spins[0]), len(out_spins[1])), axis=1)
    h = h.reshape(-1, 1, len(out_spins[0]), len(out_spins[1]))
    d_conj = get_d_matrix_lambda(angle, core_j, core_spins, out_spins[0], out_spins[1])
    return h * d_conj.reshape(-1, len(core_spins), len(out_spins[0]), len(out_spins[1]))


def compute_chain_boosts(particle_p4, chain):
    # Adapted from TF-PWA cal_chain_boost.
    # Physical meaning: boost daughters step-by-step into each mother rest frame.
    particle_set = {name for _, outs in chain for name in outs}
    core_decay_map = {}
    part_data = {}
    pending = list(chain)
    while pending:
        extra = []
        for core, outs in pending:
            if core == "Bp":
                p_rest = particle_p4[core]
                part_data[core] = {"rest_p": {}}
                for out in outs:
                    core_decay_map[out] = core
                    part_data[core]["rest_p"][out] = rest_vector(p_rest, particle_p4[out])
                    particle_set.discard(out)
                for other in list(particle_set):
                    part_data[core]["rest_p"][other] = rest_vector(p_rest, particle_p4[other])
            elif core in core_decay_map:
                parent = core_decay_map[core]
                p_rest = part_data[parent]["rest_p"][core]
                part_data[core] = {"rest_p": {}}
                for out in outs:
                    core_decay_map[out] = core
                    part_data[core]["rest_p"][out] = rest_vector(p_rest, part_data[parent]["rest_p"][out])
                    particle_set.discard(out)
                for other in list(particle_set):
                    part_data[core]["rest_p"][other] = rest_vector(p_rest, part_data[parent]["rest_p"][other])
            else:
                extra.append((core, outs))
        pending = extra
    return part_data


def calculate_helicity_angles(particle_p4, chain):
    # Adapted from TF-PWA cal_helicity_angle for this chain.
    # Physical meaning: construct the helicity angles used in Wigner-D functions.
    part_data = compute_chain_boosts(particle_p4, chain)
    set_x = {"Bp": np.array([[1.0, 0.0, 0.0]])}
    set_z = {"Bp": np.array([[0.0, 0.0, 1.0]])}
    angles = {}
    for core, outs in chain:
        angles[core] = {}
        bias = -np.pi
        for out in outs:
            z2 = part_data[core]["rest_p"][out][..., 1:]
            ang, x_axis = angle_zx_z_getx(set_z[core], set_x[core], z2)
            set_x[out] = x_axis
            set_z[out] = z2
            ang["alpha"] = (ang["alpha"] - bias) % (2 * np.pi) + bias
            bias -= np.pi
            angles[core][out] = ang
    return angles


def format_complex(z):
    z = np.asarray(z).reshape(-1)[0]
    sign = "+" if z.imag >= 0 else "-"
    return f"{z.real:.16g} {sign} {abs(z.imag):.16g}j"

# Load Input

In [3]:
# -----------------------------------------------------------------------------
# Leaf inputs: files plus the model parameters needed for the sampled-event test.
# -----------------------------------------------------------------------------

analysis_dir = Path.cwd()
if not (analysis_dir / "config_a.yml").exists():
    analysis_dir = analysis_dir / "Analysis"
print(f"Notebook working directory: {analysis_dir}")

with open(analysis_dir / "config_a.yml", "r", encoding="utf-8") as f:
    config_yml = yaml.safe_load(f)

with open(analysis_dir / "Resonances.yml", "r", encoding="utf-8") as f:
    resonances_yml = yaml.safe_load(f)

with open(analysis_dir / "final_params_full.json", "r", encoding="utf-8") as f:
    params = json.load(f)["value"]

print("Loaded config_a.yml, Resonances.yml, and final_params_full.json")

# Nominal masses from the YAML model plus Psi(4040) mass/width from the JSON parameters.
nominal_mass = {
    "Bp": float(config_yml["particle"]["$top"]["Bp"]["mass"]),
    "D": float(config_yml["particle"]["$finals"]["D"]["mass"]),
    "K": float(config_yml["particle"]["$finals"]["K"]["mass"]),
    "D0": float(config_yml["particle"]["$finals"]["D0"]["mass"]),
    "pi": float(config_yml["particle"]["$finals"]["pi"]["mass"]),
    "Dst": float(config_yml["particle"]["Dst"]["mass"]),
    "Psi(4040)": float(params["Psi(4040)_mass"]),
}
psi_width = float(params["Psi(4040)_width"])

# The script overwrites all selected Psi(4040) couplings to 1 + 0j for this isolated probe.
g_total = 1.0 + 0.0j
g_bp_to_psi_k = 1.0 + 0.0j
g_psi_to_dst_d = 1.0 + 0.0j
g_dst_to_d0_pi = 1.0 + 0.0j

print("Nominal masses:")
for name in ["Bp", "D", "K", "D0", "pi", "Dst", "Psi(4040)"]:
    print(f"  {name:>9}: {nominal_mass[name]}")
print(f"Psi(4040) width: {psi_width}")


Notebook working directory: c:\Users\gamma\Documents\Playground\B2DxDK.jl_playground\Analysis


Loaded config_a.yml, Resonances.yml, and final_params_full.json
Hardcoded four-vectors:
   D: [2.0452, -0.1467, 0.2235, -0.7847]
  D0: [2.2606, 0.2284, -0.3689, 1.2019]
   K: [0.7718, -0.0873, 0.1803, -0.5584]
  pi: [0.2017, 0.0056, -0.0349, 0.1413]
c selector: [-1.0]
Psi(4040) mass/width: 4.039 / 0.08


# Random TF-PWA Event Test

This section samples one random TF-PWA phase-space event and reruns Steps 1-8 with the isolated functions to test event-by-event consistency.


In [ ]:
print("Random event setup: generate one TF-PWA phase-space event for Bp -> D K D0 pi.")
import sys
repo_root = analysis_dir.parent
tfpwa_src = repo_root / "tf-pwa"
if str(tfpwa_src) not in sys.path:
    sys.path.insert(0, str(tfpwa_src))

import tensorflow as tf
from tf_pwa.phasespace import PhaseSpaceGenerator as TFPWAPhaseSpaceGenerator

sample_seed = int(np.random.default_rng().integers(0, 2**31 - 1))
print(f"  TF random seed: {sample_seed}")
tf.random.set_seed(sample_seed)
sample_generator = TFPWAPhaseSpaceGenerator(
    nominal_mass["Bp"],
    [nominal_mass[name] for name in ["D", "K", "D0", "pi"]],
)
sample_phase_space_list = [np.asarray(v) for v in sample_generator.generate(1)]
sampled_p4 = dict(zip(["D", "K", "D0", "pi"], sample_phase_space_list))
sampled_c_selector = np.array([-1.0])
for name in ["D", "K", "D0", "pi"]:
    print(f"  sampled {name:>2}: {sampled_p4[name][0].tolist()}")


# Random Event Execution Flow


In [ ]:
print("Random-event Step 1: Reconstruct intermediate four-vectors by summing daughters.")
sampled_p4["Dst"] = sampled_p4["D0"] + sampled_p4["pi"]
sampled_p4["Psi(4040)"] = sampled_p4["Dst"] + sampled_p4["D"]
sampled_p4["Bp"] = sampled_p4["Psi(4040)"] + sampled_p4["K"]
for name in ["Dst", "Psi(4040)", "Bp"]:
    print(f"  {name:>9}: {sampled_p4[name][0].tolist()}")


In [ ]:
print("\nRandom-event Step 2: Compute invariant masses from the event kinematics.")
sampled_event_mass = {name: invariant_mass(vec) for name, vec in sampled_p4.items()}
for name in ["Bp", "Psi(4040)", "Dst", "D", "K", "D0", "pi"]:
    print(f"  m({name}) = {sampled_event_mass[name][0]:.12f} GeV")


In [ ]:
print("\nRandom-event Step 3: Compute helicity Euler angles for the sequential decay chain.")
sampled_chain = [
    ("Bp", ["Psi(4040)", "K"]),
    ("Psi(4040)", ["Dst", "D"]),
    ("Dst", ["D0", "pi"]),
]
sampled_angles = calculate_helicity_angles(sampled_p4, sampled_chain)
for core, outs in sampled_chain:
    for out in outs:
        a = sampled_angles[core][out]
        print(f"  {core:>9} -> {out:<9}: alpha={a['alpha'][0]: .12f}, beta={a['beta'][0]: .12f}, gamma={a['gamma'][0]: .12f}")


In [ ]:
print("\nRandom-event Step 4: Compute breakup momenta q^2 and nominal q0^2 for barrier factors.")
sampled_q2_bp = get_relative_p2(sampled_event_mass["Bp"], sampled_event_mass["Psi(4040)"], sampled_event_mass["K"])
sampled_q02_bp = get_relative_p2(nominal_mass["Bp"], nominal_mass["Psi(4040)"], nominal_mass["K"])
sampled_q2_psi = get_relative_p2(sampled_event_mass["Psi(4040)"], sampled_event_mass["Dst"], sampled_event_mass["D"])
sampled_q02_psi = get_relative_p2(nominal_mass["Psi(4040)"], nominal_mass["Dst"], nominal_mass["D"])
sampled_q2_dst = get_relative_p2(sampled_event_mass["Dst"], sampled_event_mass["D0"], sampled_event_mass["pi"])
print(f"  Bp -> Psi K:      q2={sampled_q2_bp[0]:.12f}, q02={sampled_q02_bp:.12f}")
print(f"  Psi -> Dst D:     q2={sampled_q2_psi[0]:.12f}, q02={sampled_q02_psi:.12f}")
print(f"  Dst -> D0 pi:     q2={sampled_q2_dst[0]:.12f}; no barrier factor in this config")


In [ ]:
print("\nRandom-event Step 5: Build the three helicity-decay tensors.")
spin0 = (0,)
spin1 = (-1, 0, 1)
sampled_amp_bp = helicity_decay_amp(
    core_j=0,
    out_js=(1, 0),
    core_spins=spin0,
    out_spins=(spin1, spin0),
    ls_list=((1, 1),),
    g_ls=g_bp_to_psi_k,
    angle=sampled_angles["Bp"]["Psi(4040)"],
    mass_core=sampled_event_mass["Bp"],
    q2=sampled_q2_bp,
    q02=sampled_q02_bp,
    has_barrier=True,
    barrier_norm=True,
)
sampled_amp_psi = helicity_decay_amp(
    core_j=1,
    out_js=(1, 0),
    core_spins=spin1,
    out_spins=(spin1, spin0),
    ls_list=((1, 1),),
    g_ls=g_psi_to_dst_d,
    angle=sampled_angles["Psi(4040)"]["Dst"],
    mass_core=sampled_event_mass["Psi(4040)"],
    q2=sampled_q2_psi,
    q02=sampled_q02_psi,
    has_barrier=True,
    barrier_norm=True,
)
sampled_amp_dst = helicity_decay_amp(
    core_j=1,
    out_js=(0, 0),
    core_spins=spin1,
    out_spins=(spin0, spin0),
    ls_list=((1, 0),),
    g_ls=g_dst_to_d0_pi,
    angle=sampled_angles["Dst"]["D0"],
    mass_core=sampled_event_mass["Dst"],
    q2=sampled_q2_dst,
    q02=0.0,
    has_barrier=False,
    barrier_norm=False,
)
print(f"  Bp -> Psi K tensor size: {sampled_amp_bp.shape}")
print(f"  Psi -> Dst D tensor size: {sampled_amp_psi.shape}")
print(f"  Dst -> D0 pi tensor size: {sampled_amp_dst.shape}")


In [ ]:
print("\nRandom-event Step 6: Compute particle factors.")
sampled_q_psi = np.sqrt(sampled_q2_psi)
sampled_q0_psi = np.sqrt(sampled_q02_psi)
sampled_psi_factor = bwr(sampled_event_mass["Psi(4040)"], nominal_mass["Psi(4040)"], psi_width, sampled_q_psi, sampled_q0_psi, l=1, d=3.0)
sampled_dst_factor = np.ones_like(sampled_psi_factor, dtype=complex)
print(f"  Psi(4040) factor: {format_complex(sampled_psi_factor)}")
print(f"  Dst model-one factor: {format_complex(sampled_dst_factor)}")


In [ ]:
print("\nRandom-event Step 7: Contract the tensors exactly like DecayChain.get_amp.")
sampled_particle_factor = g_total * sampled_psi_factor * sampled_dst_factor
sampled_amplitude_tensor = np.einsum("...agd,...gfb,...fce,...->...abcde", sampled_amp_bp, sampled_amp_psi, sampled_amp_dst, sampled_particle_factor)
sampled_amplitude = sampled_amplitude_tensor.reshape(-1)[0]
print(f"  Isolated sampled amplitude: {format_complex(sampled_amplitude)}")


In [ ]:
print("\nRandom-event Step 8: Compare the isolated sampled amplitude with live TF-PWA.")
import os
import sys
repo_root = analysis_dir.parent
if str(repo_root / "tf-pwa") not in sys.path:
    sys.path.insert(0, str(repo_root / "tf-pwa"))
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))
os.chdir(analysis_dir)

import extra_amp
from tf_pwa.config_loader import ConfigLoader

sample_config = ConfigLoader("config_a.yml")
with open("final_params_full.json", "r", encoding="utf-8") as f:
    sample_params_dict = json.load(f)["value"]
sample_particles = list(sample_config.get_decay().outs)
sample_particle_map = {p.name: p for p in sample_particles}
sampled_p4_tfpwa = {
    sample_particle_map["D"]: tf.constant(sampled_p4["D"], dtype=tf.float64),
    sample_particle_map["D0"]: tf.constant(sampled_p4["D0"], dtype=tf.float64),
    sample_particle_map["K"]: tf.constant(sampled_p4["K"], dtype=tf.float64),
    sample_particle_map["pi"]: tf.constant(sampled_p4["pi"], dtype=tf.float64),
}
sampled_phsp_variables = sample_config.data.cal_angle(sampled_p4_tfpwa)
sampled_phsp_variables["c"] = sampled_c_selector
sample_amp_model = sample_config.get_amplitude()
sample_dg = sample_amp_model.decay_group
sample_chain = sample_dg.chains[5]
sample_p_unit = sample_params_dict.copy()
for key in sample_p_unit:
    if ("total" in key or "g_ls" in key) and (key.endswith("r") or key.endswith("i")):
        sample_p_unit[key] = 0.0
for d_idx, decay in enumerate(sample_chain.chain):
    prefix = f"{decay.core.name.replace('(1+)', '(1.)')}->{'.'.join([p.name.replace('(1+)', '(1.)') for p in decay.outs])}"
    for key in sample_p_unit:
        if prefix in key and ("total" in key or "g_ls" in key) and key.endswith("_0r"):
            sample_p_unit[key] = 1.0
sample_config.set_params(sample_p_unit)
sample_dg.set_used_chains([5])
sampled_tfpwa_live = sample_dg.get_amp(sampled_phsp_variables).numpy().reshape(-1)[0]
print(f"  Isolated sampled amplitude: {format_complex(sampled_amplitude)}")
print(f"  Live TF-PWA amplitude:    {format_complex(sampled_tfpwa_live)}")
print(f"  Sampled-event difference: {sampled_amplitude - sampled_tfpwa_live}")
assert np.abs(sampled_amplitude - sampled_tfpwa_live) < 5e-10, "Sampled-event isolated amplitude does not match live TF-PWA."
print("  PASS: sampled event also matches live TF-PWA.")
